# 2.0 — Modelos Pydantic

Ejercita los esquemas de `src/data/schemas/`: construcción válida, propiedades y
fallos de validación. La última sección enriquece vía PubChem (red) y es opcional.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # el notebook vive en notebooks/ -> raíz del repo
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from datetime import date

from src.data.schemas import ATCCode, Drug, Interaction, Med, Patient, SideEffect

## Construcción válida

Un paciente polimedicado con una medicación (amoxicilina) y su principio activo.

In [ ]:
atc = ATCCode(code="J01CA04")  # nombre resuelto desde codes.json

rash = SideEffect(
    name="rash",
    description="erupción cutánea",
    severity="mild",
)

interaction = Interaction(
    interacting_drug="warfarina",
    mechanism="aumento del efecto anticoagulante",
    management="vigilar INR",
)

amox = Drug(
    cid=33613,
    name="amoxicillin",
    chemical_group=atc,
    molecular_formula="C16H19N3O5S",
    side_effects=[rash],
    interactions=[interaction],
)

med = Med(
    ATC_code=atc,
    name="Amoxicilina 500mg",
    dosage="500 mg",
    frequency="cada 8h",
    start_date=date(2026, 6, 1),
    end_date=date(2026, 6, 10),
    active_principles=[amox],
)

patient = Patient(
    id=1,
    name="Paciente de prueba",
    age=78,
    birth_date=date(1948, 1, 1),
    number_of_meds=1,
    polymedicated=True,
    diseases=["infección respiratoria"],
    medication=[med],
)

patient.model_dump()

## Propiedades derivadas

In [ ]:
print("med.is_active :", med.is_active)   # end_date pasada -> False
print("med.duration  :", med.duration, "días")
print("atc.level     :", atc.level)
print("atc.class     :", atc.pharmacological_class)

## Fallos de validación

Cada celda provoca un error y muestra su mensaje. `ValidationError` de Pydantic
envuelve el `ValueError` lanzado por cada validador.

In [ ]:
from pydantic import ValidationError

# 1) Código ATC con formato inválido -> validation.invalid_atc_code
try:
    ATCCode(code="ZZZ99")
except ValidationError as exc:
    print(exc)

In [ ]:
# 2) Dos principios activos con el mismo CID -> unique_active_principles
try:
    Med(
        ATC_code=atc,
        name="dup",
        dosage="1",
        frequency="1/d",
        start_date=date(2026, 6, 1),
        active_principles=[
            Drug(cid=33613, name="amoxicillin"),
            Drug(cid=33613, name="amoxicillin (dup)"),
        ],
    )
except ValidationError as exc:
    print(exc)

In [ ]:
# 3) start_date posterior a end_date -> check_dates
try:
    Med(
        ATC_code=atc,
        name="fechas",
        dosage="1",
        frequency="1/d",
        start_date=date(2026, 6, 10),
        end_date=date(2026, 6, 1),
    )
except ValidationError as exc:
    print(exc)

In [ ]:
# 4) age incoherente con birth_date -> check_age_matches_birth_date
try:
    Patient(
        id=2,
        name="edad mal",
        age=40,
        birth_date=date(1948, 1, 1),  # ~78 años reales
        number_of_meds=0,
        polymedicated=False,
        diseases=[],
        medication=[],
    )
except ValidationError as exc:
    print(exc)

## Enriquecimiento vía PubChem — OPCIONAL ⚠️

`get_drug_by_cid` y `build_med` consultan la **API de PubChem** (red). No hace
falta ejecutarlas para validar el notebook. CID 33613 = amoxicilina.

In [ ]:
# from src.data.repository import build_med, get_drug_by_cid
#
# drug = get_drug_by_cid(33613)  # resuelve nombre/fórmula/SMILES/InChIKey desde PubChem
# print(drug.model_dump() if drug else "no resuelto")
#
# med_enriquecida = build_med(
#     atc_code="J01CA04",
#     name="Amoxicilina 500mg",
#     dosage="500 mg",
#     frequency="cada 8h",
#     start_date=date(2026, 6, 1),
#     cids=[33613],
# )
# med_enriquecida.model_dump()